# 03 - Indexes and `ml.bus_matching_contestedness`

Third notebook for the Bus Matching model: closes out the Section 1
precomputation work not already covered by `01_build_candidate_pairs.ipynb`
and `02_build_avl_positions.ipynb` -- a `bus_id`-first index on
`ml.trip_validity_final` (the trips table this model actually reads;
`silver.avl_pings` and the fares table already have everything the plan
asks for, confirmed against `pg_indexes` before writing this) and the
contestedness table that routes the labeling UI between uncontested
(map-only) and contested (side-by-side) mode.

**Unit here is `(bus_id, trip_date)`, valid trips only** (`is_valid`),
matching the plan's stated input population.


In [1]:
import os
from pathlib import Path

import psycopg

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## Stage 1 - `bus_id`-first index on `ml.trip_validity_final`

`trip_validity_final` only had `route_date_idx (route_id, trip_date)`
and the two shape indexes -- nothing keyed by `bus_id` first. Every
downstream step in this model (feature computation walks one date's
candidate buses; the contestedness build and later per-bus lookups) is a
`bus_id`-first access pattern, so add the composite the plan's
Section 1.1 calls for.


In [4]:
conn.execute(
    "CREATE INDEX IF NOT EXISTS trip_validity_final_bus_date_idx "
    "ON ml.trip_validity_final (bus_id, trip_date);"
)
conn.execute("ANALYZE ml.trip_validity_final;")
conn.commit()
print("trip_validity_final_bus_date_idx ready")

trip_validity_final_bus_date_idx ready


## Stage 2 - `ml.bus_matching_contestedness`

Pure AFC query over `ml.trip_validity_final`, no AVL involved. For each
valid trip, count the distinct *other* `bus_id`s running the *same*
`route_id` on the *same* `trip_date` with an overlapping
`[trip_start_timestamp, trip_end_timestamp)` window (strict interval
overlap: `t2.start < t1.end AND t2.end > t1.start`). Aggregate to
`(bus_id, trip_date)`: trip count, contested trip count, contested
fraction, the max number of simultaneously competing buses on any one
trip, and a boolean `is_contested` flag for routing the labeling UI.

Timed the self-join first via `EXPLAIN (ANALYZE, BUFFERS)` on the full
720,080-row valid-trip population before committing to the full build:
confirmed live at **9.8s** for the join+aggregate (Postgres picks a
sort-merge join on `(route_id, trip_date)`, spilling ~140MB to disk
temp files -- fine for a one-time build, not worth forcing a different
plan). No new index needed for it: `route_date_idx` already covers the
join key.


In [5]:
conn.execute("DROP TABLE IF EXISTS ml.bus_matching_contestedness;")
conn.execute(
    """
    CREATE TABLE ml.bus_matching_contestedness AS
    WITH trips AS (
        SELECT trip_id, bus_id, route_id, trip_date,
               trip_start_timestamp, trip_end_timestamp
        FROM ml.trip_validity_final
        WHERE is_valid
    ),
    overlap_counts AS (
        SELECT
            t1.trip_id,
            count(DISTINCT t2.bus_id) AS competing_buses
        FROM trips t1
        JOIN trips t2
          ON t2.route_id = t1.route_id
         AND t2.trip_date = t1.trip_date
         AND t2.bus_id <> t1.bus_id
         AND t2.trip_start_timestamp < t1.trip_end_timestamp
         AND t2.trip_end_timestamp > t1.trip_start_timestamp
        GROUP BY t1.trip_id
    )
    SELECT
        t.bus_id,
        t.trip_date,
        count(*)::integer AS n_trips,
        count(oc.trip_id)::integer AS n_contested_trips,
        (count(oc.trip_id)::float8 / count(*)) AS contested_fraction,
        coalesce(max(oc.competing_buses), 0)::integer AS max_competing_buses,
        (coalesce(max(oc.competing_buses), 0) > 0) AS is_contested
    FROM trips t
    LEFT JOIN overlap_counts oc ON oc.trip_id = t.trip_id
    GROUP BY t.bus_id, t.trip_date;
    """
)
conn.execute(
    "ALTER TABLE ml.bus_matching_contestedness ADD PRIMARY KEY (bus_id, trip_date);"
)
conn.execute("CREATE INDEX ON ml.bus_matching_contestedness (trip_date, is_contested);")
conn.execute("ANALYZE ml.bus_matching_contestedness;")
conn.commit()
print("ml.bus_matching_contestedness built")

ml.bus_matching_contestedness built


## Sanity check

In [6]:
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.bus_matching_contestedness;")
    print("bus-date rows:", cur.fetchone())

    cur.execute(
        "SELECT count(*), count(*) FILTER (WHERE is_contested) "
        "FROM ml.bus_matching_contestedness;"
    )
    print("total bus-dates / contested bus-dates:", cur.fetchone())

    cur.execute(
        "SELECT round(avg(contested_fraction)::numeric, 4), "
        "round(max(contested_fraction)::numeric, 4) "
        "FROM ml.bus_matching_contestedness;"
    )
    print("avg / max contested_fraction:", cur.fetchone())

    cur.execute(
        "SELECT max_competing_buses, count(*) FROM ml.bus_matching_contestedness "
        "GROUP BY max_competing_buses ORDER BY max_competing_buses;"
    )
    print("distribution of max_competing_buses:", cur.fetchall())

    cur.execute(
        "SELECT sum(n_trips), sum(n_contested_trips) "
        "FROM ml.bus_matching_contestedness;"
    )
    print("total trips / total contested trips:", cur.fetchone())

bus-date rows: (41332,)
total bus-dates / contested bus-dates: (41332, 39998)
avg / max contested_fraction: (Decimal('0.9347'), Decimal('1.0000'))
distribution of max_competing_buses: [(0, 1334), (1, 3037), (2, 4267), (3, 4046), (4, 2891), (5, 3506), (6, 2047), (7, 1470), (8, 1460), (9, 1128), (10, 1095), (11, 890), (12, 1224), (13, 2210), (14, 1775), (15, 1598), (16, 1392), (17, 1004), (18, 636), (19, 1342), (20, 816), (21, 709), (22, 295), (23, 31), (24, 73), (25, 759), (26, 287), (27, 10)]
total trips / total contested trips: (720080, 636635)
